### Import Packages

In [1]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

### Get Subcellular Location DataFrame

In [96]:
# Specify the file path
HPA_VERSION = "v24"
file_path = f"../hpa-datasets/{HPA_VERSION}/subcellular_location.tsv"

# Read the TSV file into a DataFrame
genes_df = pd.read_csv(file_path, sep="\t")

In [97]:
genes_df.head()

,Gene,Gene name,Reliability,Main location,Additional location,Extracellular location,Enhanced,Supported,Approved,Uncertain,Single-cell variation intensity,Single-cell variation spatial,Cell cycle dependency,GO id
0,ENSG00000000003,TSPAN6,Approved,Cell Junctions;Cytosol,Nucleoli fibrillar center,NaN,NaN,NaN,Cell Junctions;Cytosol;Nucleoli fibrillar center,NaN,Cytosol,NaN,NaN,Cell Junctions (GO:0030054);Cytosol (GO:000582...
1,ENSG00000000457,SCYL3,Supported,Cytosol;Golgi apparatus,NaN,NaN,NaN,Cytosol;Golgi apparatus,NaN,NaN,NaN,NaN,NaN,Cytosol (GO:0005829);Golgi apparatus (GO:0005794)
2,ENSG00000000460,C1orf112,Supported,Nucleoplasm,Nucleoli,NaN,NaN,Nucleoli;Nucleoplasm,NaN,NaN,NaN,NaN,NaN,Nucleoli (GO:0005730);Nucleoplasm (GO:0005654)
3,ENSG00000000938,FGR,Approved,Plasma membrane,Aggresome,NaN,NaN,NaN,Aggresome;Plasma membrane,NaN,NaN,NaN,NaN,Aggresome (GO:0016235);Plasma membrane (GO:000...
4,ENSG00000000971,CFH,Approved,Vesicles,NaN,Predicted to be secreted,NaN,NaN,Vesicles,NaN,NaN,NaN,NaN,Vesicles (GO:0043231)


### Get Target Data from Single Gene

In [98]:
# Fetch the XML
url = f"https://{HPA_VERSION}.proteinatlas.org/ENSG00000000003.xml"
response = requests.get(url)

# Parse the XML
root = ET.fromstring(response.content)

# Find the <cellExpression> element
cell_expr = root.find(".//cellExpression")

# Extract sub-elements
summary = cell_expr.find("summary").text.strip()
verification_type = cell_expr.find("verification").attrib.get("type")
verification = cell_expr.find("verification").text.strip()

image_url = cell_expr.find(".//imageUrl").text

locations = []
for loc in cell_expr.findall(".//location"):
    locations.append({
        "status": loc.attrib.get("status"),
        "GOId": loc.attrib.get("GOId"),
        "location": loc.text
    })

# Display extracted data
print("Summary:", summary)
print("Verification Type:", verification_type)
print("Verification:", verification)
print("Image URL:", image_url)
print("Locations:")
for loc in locations:
    print(f" - {loc['status']} → {loc['location']} (GO: {loc['GOId']})")

Summary: Mainly localized to the cell junctions and cytosol. In addition localized to the nucleoli fibrillar center.
Verification Type: reliability
Verification: approved
Image URL: https://images.proteinatlas.org/4109/1843_B2_17_cr5af971a263864_selected.jpg
Locations:
 - main → cell junctions (GO: GO:0030054)
 - main → cytosol (GO: GO:0005829)
 - additional → nucleoli fibrillar center (GO: GO:0001650)


In [99]:
# Fetch the XML
url = f"https://{HPA_VERSION}.proteinatlas.org/ENSG00000000003.xml"
response = requests.get(url)

# Parse the XML
root = ET.fromstring(response.content)

# Find the cellExpression block
cell_expr = root.findall(".//cellExpression")[1]

# Print cell_expr content
# for child in cell_expr:
#     print(child.tag, child.attrib)

# Get verification type
verification = cell_expr.find(".//verification")
verification_type = verification.attrib.get("type") if verification is not None else None
verification = verification.text.strip() if verification is not None else None

# All <data> entries
results = []
for data in cell_expr.findall(".//data"):
    # print("\nData block:")
    # for child in data:
    #     print(" -", child.tag, child.attrib)
    cell_line_elem = data.find("cellLine")
    cell_line = cell_line_elem.text if cell_line_elem is not None else None
    organ = cell_line_elem.attrib.get("organ") if cell_line_elem is not None else None
    cell_id = cell_line_elem.attrib.get("cellosaurusID") if cell_line_elem is not None else None

    # Locations
    locations = [loc.text for loc in data.findall("location")]

    # Sample images (without channel info)
    image_urls = [
        img.find("imageUrl").text
        for img in data.findall(".//image[@imageType='sampleImage']")
        if img.find("imageUrl") is not None
    ]

    results.append({
        "Cell Line": cell_line,
        "Organ": organ,
        "Cellosaurus ID": cell_id,
        "Locations": locations,
        "Images": image_urls
    })

# Output example
print("Verification Type:", verification_type)
print("Verification:", verification)
for res in results:
    print("\nCell Line:", res["Cell Line"])
    print("Organ:", res["Organ"])
    print("Cellosaurus ID:", res["Cellosaurus ID"])
    print("Locations:", ", ".join(res["Locations"]))
    print("Images:")
    for img in res["Images"]:
        print(" -", img)

Verification Type: validation
Verification: approved

Cell Line: CACO-2
Organ: Gastrointestinal tract
Cellosaurus ID: CVCL_0025
Locations: nucleoli fibrillar center, cytosol
Images:
 - https://images.proteinatlas.org/4109/1832_C1_2_blue_red_green.jpg
 - https://images.proteinatlas.org/4109/1832_C1_4_blue_red_green.jpg

Cell Line: RT-4
Organ: Kidney & Urinary bladder
Cellosaurus ID: CVCL_0036
Locations: nucleoli fibrillar center, cell junctions, cytosol
Images:
 - https://images.proteinatlas.org/4109/1843_B2_17_cr5af971a263864_blue_red_green.jpg
 - https://images.proteinatlas.org/4109/1843_B2_30_cr5af971a2648d6_blue_red_green.jpg

Cell Line: U2OS
Organ: Mesenchymal
Cellosaurus ID: CVCL_0042
Locations: cytosol
Images:
 - https://images.proteinatlas.org/4109/23_H11_1_blue_red_green.jpg
 - https://images.proteinatlas.org/4109/23_H11_2_blue_red_green.jpg


### Get Target Data from List of Genes

In [21]:
def get_hpa_data_from_gene(gene_id, hpa_version):
    # Fetch the XML
    url = f"https://{hpa_version}.proteinatlas.org/{gene_id}.xml"
    response = requests.get(url)

    # Parse the XML
    root = ET.fromstring(response.content)

    # Find the cellExpression block
    cell_expr = root.findall(".//cellExpression")
    cell_expr_0 = cell_expr[0] if cell_expr else None
    cell_expr_1 = cell_expr[1] if len(cell_expr) > 1 else None

    # Extract sub-elements
    gen_source = cell_expr_0.attrib.get("source")
    gen_tech = cell_expr_0.attrib.get("technology")
    gen_summary = cell_expr_0.find("summary").text.strip()
    gen_verification_type = cell_expr_0.find("verification").attrib.get("type")
    gen_verification = cell_expr_0.find("verification").text.strip()

    gen_image_url = cell_expr_0.find(".//imageUrl").text

    gen_locations = []
    for loc in cell_expr_0.findall(".//location"):
        gen_locations.append({
            "status": loc.attrib.get("status"),
            "GOId": loc.attrib.get("GOId"),
            "location": loc.text
        })

    # Get verification type
    source = cell_expr_1.attrib.get("source")
    tech = cell_expr_1.attrib.get("technology")
    verification = cell_expr_1.find(".//verification")
    verification_type = verification.attrib.get("type") if verification is not None else None
    verification = verification.text.strip() if verification is not None else None

    image_url = cell_expr_1.find(".//imageUrl").text

    # All <data> entries
    results = []
    for data in cell_expr_1.findall(".//data"):
        cell_line_elem = data.find("cellLine")
        cell_line = cell_line_elem.text if cell_line_elem is not None else None
        organ = cell_line_elem.attrib.get("organ") if cell_line_elem is not None else None
        cell_id = cell_line_elem.attrib.get("cellosaurusID") if cell_line_elem is not None else None

        # Locations
        locations = [
            loc.text + " (" + loc.attrib.get('GOId', '') + ")" for loc in data.findall("location")]

        # Sample images (without channel info)
        image_urls = [
            img.find("imageUrl").text
            for img in data.findall(".//image[@imageType='sampleImage']")
            if img.find("imageUrl") is not None
        ]

        results.append({
            "Cell Line": cell_line,
            "Organ": organ,
            "Cellosaurus ID": cell_id,
            "Locations": locations,
            "Images": image_urls
        })

    # List of dictionaries for each image
    hpa_data = []
    for res in results:
        for img in res["Images"]:
            hpa_data.append({
                "Gene ID": gene_id,
                "General Summary": gen_summary,
                "General Source": gen_source,
                "General Technology": gen_tech,
                "General Verification Type": gen_verification_type,
                "General Verification": gen_verification,
                "General Image URL": gen_image_url,
                "Source": source,
                "Technology": tech,
                "Verification Type": verification_type,
                "Verification": verification,
                "Image URL": image_url,
                "Cell Line": res["Cell Line"],
                "Organ": res["Organ"],
                "Cellosaurus ID": res["Cellosaurus ID"],
                "Location": res["Locations"],
                "Image": img
            })

    return pd.DataFrame(hpa_data)

In [22]:
from tqdm import tqdm  # better visuals in Jupyter

def get_hpa_data(genes):
    """Fetch HPA data for a list of genes."""
    dfs = []
    for gene in tqdm(genes, desc="Fetching HPA data"):
        df = get_hpa_data_from_gene(gene_id=gene, hpa_version=HPA_VERSION)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

In [104]:
df = get_hpa_data(genes=genes_df['Gene'].to_list())

Fetching HPA data: 100%|██████████| 13534/13534 [7:08:41<00:00,  1.90s/it]  


In [105]:
df.to_csv(HPA_VERSION + "_hpa_subcellular_location_expanded.csv", index=False)

### HPAv24 vs Kaggle Public HPA Data

#### Get Kaggle Public HPA Data

In [40]:
# Used on HPA Competition training and test sets
celllines = [
    'A-431', 
    'A549', 
    'EFO-21', 
    'HAP1', 
    'HEK 293', 
    'HUVEC TERT2', 
    'HaCaT', 
    'HeLa', 
    'PC-3', 
    'RH-30', 
    'RPTEC TERT1', 
    'SH-SY5Y', 
    'SK-MEL-30', 
    'SiHa', 
    'U-2 OS', 
    'U-251 MG', 
    'hTCEpi']

In [41]:
pub_kaggle_df = pd.read_csv("../hpa-datasets/public_kaggle_2021/kaggle_2021.tsv")
pub_kaggle_df.set_index('Image', inplace=True)

In [42]:
print(len(pub_kaggle_df), "rows of public Kaggle data fetched.")
pub_kaggle_df.head(10)

82495 rows of public Kaggle data fetched.


,Label,Cellline,in_trainset,Label_idx
Image,,,,
https://images.proteinatlas.org/10005/921_B9_1,Cytosol,A-431,False,16
https://images.proteinatlas.org/10005/921_B9_2,Cytosol,A-431,False,16
https://images.proteinatlas.org/10005/923_B9_1,Cytosol,U-251 MG,False,16
https://images.proteinatlas.org/10005/923_B9_2,Cytosol,U-251 MG,False,16
https://images.proteinatlas.org/10005/931_B9_1,Cytosol,U-2 OS,False,16
https://images.proteinatlas.org/10005/931_B9_2,Cytosol,U-2 OS,False,16
https://images.proteinatlas.org/10007/1876_A7_32,Endoplasmic reticulum,ASC TERT1,False,6
https://images.proteinatlas.org/10007/1876_A7_37,Endoplasmic reticulum,ASC TERT1,False,6
https://images.proteinatlas.org/10007/1901_A10_2,Vesicles,U-2 OS,False,17


#### Get HPAv24

In [43]:
df = pd.read_csv("v24_hpa_subcellular_location_expanded.csv")

print(len(df), "rows of HPA data fetched.")
df = df[['Gene ID', 'Cell Line', 'Location', 'Image']]
df['Image'] = df['Image'].apply(lambda x: x.split('_blue_red_green.jpg')[0])
df.set_index('Image', inplace=True)
df.head(10)

83360 rows of HPA data fetched.


,Gene ID,Cell Line,Location
Image,,,
https://images.proteinatlas.org/4109/1832_C1_2,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy..."
https://images.proteinatlas.org/4109/1832_C1_4,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy..."
https://images.proteinatlas.org/4109/1843_B2_17_cr5af971a263864,ENSG00000000003,RT-4,"['nucleoli fibrillar center (GO:0001650)', 'ce..."
https://images.proteinatlas.org/4109/1843_B2_30_cr5af971a2648d6,ENSG00000000003,RT-4,"['nucleoli fibrillar center (GO:0001650)', 'ce..."
https://images.proteinatlas.org/4109/23_H11_1,ENSG00000000003,U2OS,['cytosol (GO:0005829)']
https://images.proteinatlas.org/4109/23_H11_2,ENSG00000000003,U2OS,['cytosol (GO:0005829)']
https://images.proteinatlas.org/72383/2038_G11_1,ENSG00000000457,OE19,"['golgi apparatus (GO:0005794)', 'cytosol (GO:..."
https://images.proteinatlas.org/72383/2038_G11_2,ENSG00000000457,OE19,"['golgi apparatus (GO:0005794)', 'cytosol (GO:..."
https://images.proteinatlas.org/72383/2142_F7_1,ENSG00000000457,SK-MEL-30,[]


#### Check for Different Images

In [44]:
exclusive_indexes = pub_kaggle_df.index.difference(df.index)
pub_kaggle_df.loc[exclusive_indexes]

,Label,Cellline,in_trainset,Label_idx
Image,,,,
https://images.proteinatlas.org/10021/1891_N5_28,Vesicles,SiHa,False,17
https://images.proteinatlas.org/10021/1891_N5_5,Vesicles,SiHa,False,17
https://images.proteinatlas.org/10052/945_E10_1,No staining,U-2 OS,False,18
https://images.proteinatlas.org/10052/945_E10_2,No staining,U-2 OS,False,18
https://images.proteinatlas.org/10052/946_E10_1,Mitochondria,A-431,False,14
...,...,...,...,...
https://images.proteinatlas.org/9931/923_F8_2,"Nucleoplasm,Cytosol",U-251 MG,False,0|16
https://images.proteinatlas.org/9931/931_F8_3,Nucleoplasm,U-2 OS,False,0
https://images.proteinatlas.org/9931/931_F8_4,Nucleoplasm,U-2 OS,False,0


In [45]:
exclusive_indexes = df.index.difference(pub_kaggle_df.index)
df.loc[exclusive_indexes]

,Gene ID,Cell Line,Location
Image,,,
https://images.proteinatlas.org/10021/1891_N5_28_cr5bbca45356736,ENSG00000111799,SiHa,['vesicles (GO:0043231)']
https://images.proteinatlas.org/10021/1891_N5_5_cr5bbca45356284,ENSG00000111799,SiHa,['vesicles (GO:0043231)']
https://images.proteinatlas.org/10025/si20_E1_2,ENSG00000131871,U2OS,[]
https://images.proteinatlas.org/10025/si20_E1_3,ENSG00000131871,U2OS,[]
https://images.proteinatlas.org/10025/si20_E2_2,ENSG00000131871,U2OS,[]
...,...,...,...
https://images.proteinatlas.org/9985/2136_E4_42,ENSG00000121879,RPTEC/TERT1,"['microtubules (GO:0015630)', 'primary cilium ..."
https://images.proteinatlas.org/9985/2151_G6_36,ENSG00000121879,hTERT-RPE1 (serum starved),"['primary cilium (GO:0005929)', 'cytosol (GO:0..."
https://images.proteinatlas.org/9985/2151_G6_48,ENSG00000121879,hTERT-RPE1 (serum starved),"['primary cilium (GO:0005929)', 'cytosol (GO:0..."


### HPAv20 [Retrived Data] x Kaggle Public HPA Data

#### Get HPAv20

In [49]:
df_v20 = pd.read_csv("v20_hpa_subcellular_location_expanded.csv")

print(len(df_v20), "rows of HPA data fetched.")
df_v20 = df_v20[['Gene ID', 'Cell Line', 'Location', 'Image']]
df_v20['Image'] = df_v20['Image'].apply(lambda x: x.split('_blue_red_green.jpg')[0])
df_v20['Image'] = df_v20['Image'].apply(lambda x: x.replace('http', 'https'))
df_v20.set_index('Image', inplace=True)
df_v20.head(10)

73763 rows of HPA data fetched.


,Gene ID,Cell Line,Location
Image,,,
https://images.proteinatlas.org/4109/1832_C1_2,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy..."
https://images.proteinatlas.org/4109/1832_C1_4,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy..."
https://images.proteinatlas.org/4109/1843_B2_17_cr5af971a263864,ENSG00000000003,RT4,"['nucleoli fibrillar center (GO:0001650)', 'ce..."
https://images.proteinatlas.org/4109/1843_B2_30_cr5af971a2648d6,ENSG00000000003,RT4,"['nucleoli fibrillar center (GO:0001650)', 'ce..."
https://images.proteinatlas.org/4109/23_H11_1,ENSG00000000003,U-2 OS,['cytosol (GO:0005829)']
https://images.proteinatlas.org/4109/23_H11_2,ENSG00000000003,U-2 OS,['cytosol (GO:0005829)']
https://images.proteinatlas.org/5624/69_H11_1,ENSG00000000457,A-431,"['nuclear bodies (GO:0016604)', 'microtubules ..."
https://images.proteinatlas.org/5624/69_H11_2,ENSG00000000457,A-431,"['nuclear bodies (GO:0016604)', 'microtubules ..."
https://images.proteinatlas.org/5624/91_H11_1,ENSG00000000457,U-2 OS,['microtubules (GO:0015630)']


#### Check for Different Images

In [50]:
exclusive_indexes = pub_kaggle_df.index.difference(df_v20.index)
pub_kaggle_df.loc[exclusive_indexes]

,Label,Cellline,in_trainset,Label_idx
Image,,,,
https://images.proteinatlas.org/10021/1891_N5_28,Vesicles,SiHa,False,17
https://images.proteinatlas.org/10021/1891_N5_5,Vesicles,SiHa,False,17
https://images.proteinatlas.org/10052/945_E10_1,No staining,U-2 OS,False,18
https://images.proteinatlas.org/10052/945_E10_2,No staining,U-2 OS,False,18
https://images.proteinatlas.org/10052/946_E10_1,Mitochondria,A-431,False,14
...,...,...,...,...
https://images.proteinatlas.org/9931/923_F8_2,"Nucleoplasm,Cytosol",U-251 MG,False,0|16
https://images.proteinatlas.org/9931/931_F8_3,Nucleoplasm,U-2 OS,False,0
https://images.proteinatlas.org/9931/931_F8_4,Nucleoplasm,U-2 OS,False,0


In [51]:
exclusive_indexes = df_v20.index.difference(pub_kaggle_df.index)
df_v20.loc[exclusive_indexes]

,Gene ID,Cell Line,Location
Image,,,
https://images.proteinatlas.org/10021/1891_N5_28_cr5bbca45356736,ENSG00000111799,SiHa,['vesicles (GO:0043231)']
https://images.proteinatlas.org/10021/1891_N5_5_cr5bbca45356284,ENSG00000111799,SiHa,['vesicles (GO:0043231)']
https://images.proteinatlas.org/10025/si20_E1_2,ENSG00000131871,U-2 OS,[]
https://images.proteinatlas.org/10025/si20_E1_3,ENSG00000131871,U-2 OS,[]
https://images.proteinatlas.org/10025/si20_E2_2,ENSG00000131871,U-2 OS,[]
...,...,...,...
https://images.proteinatlas.org/9975/si20_E5_3,ENSG00000137824,U-2 OS,[]
https://images.proteinatlas.org/9975/si20_E6_2,ENSG00000137824,U-2 OS,[]
https://images.proteinatlas.org/9975/si20_E6_4,ENSG00000137824,U-2 OS,[]


### Empty Location Problem [Negative Class Haha]

In [19]:
df[df.Location=='[]']

,Gene ID,Cell Line,Location
Image,,,
https://images.proteinatlas.org/72383/2142_F7_1,ENSG00000000457,SK-MEL-30,[]
https://images.proteinatlas.org/72383/2142_F7_2,ENSG00000000457,SK-MEL-30,[]
https://images.proteinatlas.org/2024/60_H5_1,ENSG00000000938,A-431,[]
https://images.proteinatlas.org/2024/60_H5_2,ENSG00000000938,A-431,[]
https://images.proteinatlas.org/38922/603_A5_1,ENSG00000000971,U2OS,[]
...,...,...,...
https://images.proteinatlas.org/41189/584_F6_2,ENSG00000288859,U-251MG,[]
https://images.proteinatlas.org/59543/1282_B2_2,ENSG00000291316,HeLa,[]
https://images.proteinatlas.org/59543/1282_B2_4,ENSG00000291316,HeLa,[]


In [24]:
single_gene_df = get_hpa_data_from_gene('ENSG00000000457', 'v24')
single_gene_df.head(10)

,Gene ID,General Summary,General Source,General Technology,General Verification Type,General Verification,General Image URL,Source,Technology,Verification Type,Verification,Image URL,Cell Line,Organ,Cellosaurus ID,Location,Image
0,ENSG00000000457,Localized to the Golgi apparatus and cytosol.,HPA,ICC/IF,reliability,supported,https://images.proteinatlas.org/72383/2038_G11...,HPA,ICC/IF,validation,supported,https://images.proteinatlas.org/72383/2038_G11...,OE19,Proximal digestive tract,CVCL_1622,"[golgi apparatus (GO:0005794), cytosol (GO:000...",https://images.proteinatlas.org/72383/2038_G11...
1,ENSG00000000457,Localized to the Golgi apparatus and cytosol.,HPA,ICC/IF,reliability,supported,https://images.proteinatlas.org/72383/2038_G11...,HPA,ICC/IF,validation,supported,https://images.proteinatlas.org/72383/2038_G11...,OE19,Proximal digestive tract,CVCL_1622,"[golgi apparatus (GO:0005794), cytosol (GO:000...",https://images.proteinatlas.org/72383/2038_G11...
2,ENSG00000000457,Localized to the Golgi apparatus and cytosol.,HPA,ICC/IF,reliability,supported,https://images.proteinatlas.org/72383/2038_G11...,HPA,ICC/IF,validation,supported,https://images.proteinatlas.org/72383/2038_G11...,SK-MEL-30,Skin,CVCL_0039,[],https://images.proteinatlas.org/72383/2142_F7_...
3,ENSG00000000457,Localized to the Golgi apparatus and cytosol.,HPA,ICC/IF,reliability,supported,https://images.proteinatlas.org/72383/2038_G11...,HPA,ICC/IF,validation,supported,https://images.proteinatlas.org/72383/2038_G11...,SK-MEL-30,Skin,CVCL_0039,[],https://images.proteinatlas.org/72383/2142_F7_...
4,ENSG00000000457,Localized to the Golgi apparatus and cytosol.,HPA,ICC/IF,reliability,supported,https://images.proteinatlas.org/72383/2038_G11...,HPA,ICC/IF,validation,supported,https://images.proteinatlas.org/72383/2038_G11...,SuSa,Male tissues,CVCL_L280,"[endoplasmic reticulum (GO:0005783), golgi app...",https://images.proteinatlas.org/72383/2091_H8_...
5,ENSG00000000457,Localized to the Golgi apparatus and cytosol.,HPA,ICC/IF,reliability,supported,https://images.proteinatlas.org/72383/2038_G11...,HPA,ICC/IF,validation,supported,https://images.proteinatlas.org/72383/2038_G11...,SuSa,Male tissues,CVCL_L280,"[endoplasmic reticulum (GO:0005783), golgi app...",https://images.proteinatlas.org/72383/2091_H8_...
